# ApproxIoT replication — results analysis

Loads the CSVs produced by `python -m experiments.run_all` from `results/`
and redraws the paper-figure analogues (Figs. 6-12) plus the novel
controller comparison figures.  Rerun the experiment suite first if the
CSVs are missing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("results")

exp1 = pd.read_csv(RESULTS / "exp1_sampling_fraction.csv") if (RESULTS / "exp1_sampling_fraction.csv").exists() else None
exp2 = pd.read_csv(RESULTS / "exp2_window_size.csv") if (RESULTS / "exp2_window_size.csv").exists() else None
exp3 = pd.read_csv(RESULTS / "exp3_fluctuating_rates.csv") if (RESULTS / "exp3_fluctuating_rates.csv").exists() else None
exp4 = pd.read_csv(RESULTS / "exp4_skew.csv") if (RESULTS / "exp4_skew.csv").exists() else None
exp5 = pd.read_csv(RESULTS / "exp5_real_world.csv") if (RESULTS / "exp5_real_world.csv").exists() else None
exp6 = pd.read_csv(RESULTS / "exp6_adaptive_controller.csv") if (RESULTS / "exp6_adaptive_controller.csv").exists() else None
frac_hist = pd.read_csv(RESULTS / "exp6_fraction_history.csv") if (RESULTS / "exp6_fraction_history.csv").exists() else None
print("loaded:", [n for n,df in [("exp1",exp1),("exp2",exp2),("exp3",exp3),("exp4",exp4),("exp5",exp5),("exp6",exp6)] if df is not None])

## EXP1 — accuracy loss & forwarded volume vs sampling fraction (Figs. 6-9 analogue)

In [ ]:
if exp1 is not None:
    g = exp1[exp1.method != "native"].groupby(["method", "fraction"])["realized_loss"].median().unstack("method")
    fig, ax = plt.subplots(figsize=(7, 4))
    g.plot(marker="o", logy=True, ax=ax)
    ax.set_xlabel("sampling fraction")
    ax.set_ylabel("median realized loss")
    ax.set_title("EXP1: accuracy loss vs fraction (Gaussian)")
    ax.grid(alpha=0.3)
    plt.show()
    display(g)

## EXP2 — e2e latency vs window size (Fig. 10 analogue)

In [ ]:
if exp2 is not None:
    display(exp2)

## EXP3 — fluctuating rate settings (§V-D analogue)

In [ ]:
if exp3 is not None:
    piv = exp3.groupby(["phase", "method"])["realized_loss"].median().unstack("method")
    display(piv)
    piv.plot(marker="o", logy=True, figsize=(7, 4),
             title="EXP3: median loss per rate-setting phase")

## EXP4 — extreme skew (§V-E analogue, the dramatic gap)

In [ ]:
if exp4 is not None:
    piv = exp4.groupby(["fraction", "method"])["realized_loss"].median().unstack("method")
    display(piv)
    piv.plot(marker="o", logy=True, figsize=(7, 4),
             title="EXP4: skew workload, ApproxIoT vs SRS (log scale)")

## EXP5 — real-world datasets (§VI analogue)

In [ ]:
if exp5 is not None:
    multi_path = RESULTS / "exp5_real_world_multiseed.csv"
    src = pd.read_csv(multi_path) if multi_path.exists() else exp5
    piv = src.groupby(["fraction", "method"])["realized_loss"].median().unstack("method")
    display(piv)
    piv.plot(marker="o", logy=True, figsize=(7, 4),
             title="EXP5: taxi + air-quality workload")

## EXP6 — the novelty: adaptive controller vs static budget

Error signal over time (top row of the saved figure) and the sampling
fraction the controller issued (bottom); the vertical dotted line marks the
mid-run rate step-change.

In [ ]:
from IPython.display import Image, display
p = RESULTS / "exp6_adaptive_controller.png"
if p.exists():
    display(Image(filename=str(p)))
if exp6 is not None:
    display(exp6[exp6.scenario != "skew"].groupby(["scenario", "regime"])[
        ["realized_loss", "bandwidth_saved"]].median())

### Verdicts

In [ ]:
import json
v = RESULTS / "summary.json"
if v.exists():
    summary = json.loads(v.read_text())
    for name, payload in summary.items():
        check = payload.get("qualitative_check") if isinstance(payload, dict) else None
        if check is not None:
            print(f"{name}: pass={check.get('pass')}  ({ {k: x for k,x in check.items() if k != 'pass'} })")
else:
    print("run `python -m experiments.run_all` first")